# Image Analysis

Explore image processing results from the Social Explorer pipeline: object detection,
face detection, and image classification tags.

**What this notebook covers:**
- Image processing summary statistics
- Object detection results and top detected objects
- Face detection distribution
- Classification tag comparison (heuristic vs CLIP)
- LLM-assisted pattern analysis

In [ ]:
from workspace_utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Image Processing Summary

In [ ]:
# Overall image analysis summary
summary = image_analysis_summary()
print('Image Analysis Summary')
summary

## 2. Object Detection Results

In [ ]:
# Query object detection results from the database
objects_df = query_df("""
    SELECT object_class, COUNT(*) as count
    FROM image_objects
    GROUP BY object_class
    ORDER BY count DESC
""")

print(f'Found {len(objects_df)} unique object classes')
objects_df.head(10)

In [ ]:
# Bar chart of top detected objects
top_objects = objects_df.head(20)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(
    top_objects['object_class'],
    top_objects['count'],
    color=sns.color_palette('magma', len(top_objects))
)
ax.set_xlabel('Detection Count')
ax.set_title('Top 20 Detected Objects')
ax.invert_yaxis()

# Add count labels
for bar, count in zip(bars, top_objects['count']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Face Detection Distribution

In [ ]:
# Query face detection results
faces_df = query_df("""
    SELECT face_count, COUNT(*) as image_count
    FROM image_faces
    GROUP BY face_count
    ORDER BY face_count
""")

print(f'Face count distribution across {faces_df["image_count"].sum()} images')
faces_df

In [ ]:
# Histogram of face counts
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    faces_df['face_count'].astype(str),
    faces_df['image_count'],
    color=sns.color_palette('coolwarm', len(faces_df))
)
ax.set_xlabel('Number of Faces Detected')
ax.set_ylabel('Number of Images')
ax.set_title('Face Count Distribution')
plt.tight_layout()
plt.show()

## 4. Classification Tags: Heuristic vs CLIP

In [ ]:
# Query classification tags by model
tags_df = query_df("""
    SELECT tag, COUNT(*) as count, model
    FROM image_classification
    GROUP BY tag, model
    ORDER BY count DESC
    LIMIT 20
""")

print(f'Top classification tags by model')
tags_df

In [ ]:
# Compare heuristic vs CLIP tags
models = tags_df['model'].unique()
n_models = len(models)

fig, axes = plt.subplots(1, max(n_models, 1), figsize=(7 * max(n_models, 1), 6))
if n_models == 1:
    axes = [axes]

for ax, model in zip(axes, models):
    subset = tags_df[tags_df['model'] == model].head(10)
    ax.barh(subset['tag'], subset['count'], color=sns.color_palette('Set2', len(subset)))
    ax.set_title(f'Top Tags: {model}')
    ax.set_xlabel('Count')
    ax.invert_yaxis()

plt.suptitle('Image Classification Tags by Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. LLM Pattern Analysis

In [ ]:
# Ask the LLM to identify visual content themes
analysis = analyze_dataframe(
    objects_df,
    "What visual content themes dominate? What do the most common objects tell us "
    "about the type of imagery being shared on these social platforms?"
)
print(analysis)

In [ ]:
# Summary
print('Image Analysis Complete!')
print(f'  Unique object classes: {len(objects_df)}')
print(f'  Images with face data: {faces_df["image_count"].sum()}')
print(f'  Classification models: {models.tolist()}')
print('\nNext: try cross_platform_analytics.ipynb or topic_modeling.ipynb')